In [ ]:
import pandas as pd
import glob
import os

# Get the file paths for all sequence and filter check files
all_sequences_files = glob.glob('all_sequences_*.csv')
filter_check_files = glob.glob('filter_check*.csv')

# --- Combine all sequence information into one DataFrame ---
all_sequences_dfs = []
for file in all_sequences_files:
    df = pd.read_csv(file)
    all_sequences_dfs.append(df)

# Concatenate all dataframes and drop duplicate sequences
all_sequences_df = pd.concat(all_sequences_dfs, ignore_index=True)
all_sequences_df.drop_duplicates(subset='sequence', keep='first', inplace=True)

# --- Loop through each filter_check file to process it ---
for file in filter_check_files:
    # Read the filter_check file
    filter_df = pd.read_csv(file)

    # Merge to add the 'architecture' column
    merged_df = pd.merge(filter_df, all_sequences_df[['sequence', 'architecture']], on='sequence', how='left')

    # --- NEW: Select only the first 10 samples ---
    top_10_df = merged_df.head(10)

    # --- NEW: Create a new filename to avoid overwriting ---
    # This splits the original filename from its extension (e.g., 'filter_check1' and '.csv')
    base_name = os.path.splitext(file)[0]
    new_filename = f"{base_name}_top10.csv"

    # Save the new DataFrame with the top 10 samples to the new file
    top_10_df.to_csv(new_filename, index=False)

    print(f"Saved the top 10 samples from {file} to {new_filename}")